# M07b — prompt-variant comparison (P0/P1/P2) on the request bank v2

**Thin launcher.** All logic lives in the repo behind a CLI:

    python -m pii_redteam.cli compare-prompts --config configs/m07b_prompt_v2.yaml

Question: can prompting ALONE make the base Qwen3.5-2B (no adapter, non-thinking)
produce varied, realistic short messages instead of "Name said..." /
"Hello Name, how are you?" — or is SFT v2 on Spark-written short data required?

Compares three prompt variants on the SAME committed v2 bank (20 requests, new
sha256; frozen v1 bank untouched), n=8 per request, t in {1.0, 0.7}, against
pre-committed pass criteria in the config. Every raw output is saved
(samples_raw.jsonl) so the run can be re-scored later. D0 p_target/live-group
stats are computed but flagged "pending scorer verification".

Expected runtime on T4: 3 variants x 2 temps x 20 requests x 8 samples = 960
generations at max_new_tokens=192, roughly 20-40 min.


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
# Until feat/request-v2-prompting is merged, set REF to that branch.
REF = "feat/request-v2-prompting"

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Always sync to the target ref: a reused VM may hold a STALE clone.
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "fetch", "-q", "origin", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-q", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", REF, REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True
)

import torch

free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb >= 10, (
    f"GPU has only {free_gb:.1f} GB free - dirty runtime. Runtime -> Disconnect and"
    " delete runtime, then reopen from GitHub and Run all."
)
print(f"GPU free: {free_gb:.1f} GB - OK ({torch.cuda.get_device_name(0)})")
os.environ["PROBE_REPO_COMMIT"] = commit

In [ ]:
import os
import subprocess

# Colab-specific config overrides: Drive D0 + Drive work_dir + repo_dir.
with open(os.path.join(REPO_DIR, "configs/m07b_prompt_v2.yaml"), encoding="utf-8") as f:
    config_text = f.read()
config_text = (
    config_text.replace("repo_dir: null", "repo_dir: /content/pii-redteam-rl")
    .replace("work_dir: artifacts/m07b", "work_dir: /content/drive/MyDrive/pii-redteam-rl")
    .replace(
        "path: /tmp/d0zip/pii-redteam-rl/d0-20261004-142714/d0-best",
        "path: /content/drive/MyDrive/pii-redteam-rl/d0-20261004-142714/d0-best",
    )
)
colab_config = os.path.join(REPO_DIR, "configs/m07b_prompt_v2_colab.yaml")
with open(colab_config, "w", encoding="utf-8") as f:
    f.write(config_text)

env = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src"))
print("Running comparison (progress + projection printed per cell)...")
result = subprocess.run(
    [
        "python",
        "-m",
        "pii_redteam.cli",
        "compare-prompts",
        "--config",
        "configs/m07b_prompt_v2_colab.yaml",
    ],
    cwd=REPO_DIR,
    env=env,
)
print(f"\ncomparison exit code: {result.returncode}")
if result.returncode != 0:
    print("^^ the traceback above is the real error - paste it back to me")

**When the run finishes:** the PASS BY CELL table and the recommendation are printed
above, plus 10 random valid samples per variant (t=1.0) for manual review.
Bring back `m07b-<timestamp>/prompt_compare_report.json` and `samples_raw.jsonl`
from Drive — the report's `experiments_entry` is appended to `EXPERIMENTS.yaml`
as-is and the numbers fill `docs/PROMPT_V2_RESULTS.md`. If NO variant passes,
that is the agreed trigger for SFT v2 on Spark-written short data — not more
prompt hacking.
